In [99]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math

init_printing(use_latex=True)

In [100]:
import re
from IPython.display import display, Math

def color_terms(latex_str):
    latex_str = re.sub(
        r'(\\phi_\{REF\}\{\\left\(.+?\\right\)\})',
        r'{\\color{purple} \1}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([1])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{red} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([2])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([3])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{yellow} \1{\\left(\3\\right)}}',
        latex_str
    )
    return latex_str

In [101]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

In [102]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')
phiREF  = Function(r'\phi_{REF}')   # reference laser phase (shared)

# q_i: timing / clock variables (Moku-like, referenced to global time)
q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')

# q_ref: reference laser timing variable
q_ref = Function('q_{ref}')

# Modulation frequency symbol shorthand  omega_j^m → use per-sc values

In [103]:
def D(expr, tau):
    return expr.subs(t, t - tau)

# Map spacecraft index → (phi, q, omega, omega_m)
sc = {
    1: (phi1, q1, omega1, omega1m),
    2: (phi2, q2, omega2, omega2m),
    3: (phi3, q3, omega3, omega3m),
}

tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

## Sideband phase measurement model

The sideband phase measurement on spacecraft $i$ receiving from spacecraft $j$ is:

$$
\eta_{ij}^{SB} =
\left(\delta\phi_j(t-\tau_{ji}) - \delta\phi_r(t-\tau_{ji})
      - \omega_j^m q_j(t-\tau_{ji}) + \omega_r^m q_r(t-\tau_{ji})\right)
-
\left(\delta\phi_i(t) - \delta\phi_r(t) + \omega_r^m q_r(t)\right)
- \left((\omega_i - \omega_j) + \omega_j^m\right) q_i(t)
$$

where $\omega_i - \omega_j$ is the heterodyne frequency (carrier frequency difference between local and incoming laser). Only laser phase noise ($\phi$) and timing ($q$) terms are kept.

In [104]:
# Reference modulation frequency (same symbol for all SCs, adjust if different)
omega_rm = Symbol(r'\omega_r^m', real=True)

eta    = {}
etaSB  = {}
etaLSB  = {}
include_phi = False 
include_clock_noise = True
include_REF_laser = True

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i = sc[i]
    phi_j, q_j, om_j, omm_j = sc[j]
    t_ij = tau[(i, j)]

    phi_terms = (D(phi_j(t)-int(include_REF_laser)*phiREF(t), t_ij) - (phi_i(t)-int(include_REF_laser)*phiREF(t))) if include_phi else 0

    clock_terms_C = (- (om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (- (om_j - om_i + omm_j - omm_i) * q_i(t) - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0
    clock_terms_LSB = -(- (om_i - om_j + omm_j - omm_i) * q_i(t) - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0
    
    ref_terms = omega_rm*(D(q_j(t), t_ij) - q_i(t))
    ref_termsLSB = -omega_rm*(D(q_j(t), t_ij) - q_i(t))

    eta[(i,j)] = (phi_terms + clock_terms_C)

    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + ref_terms), [q_i(t), q_j(t)])
    etaLSB[(i,j)] = collect(expand(phi_terms + clock_terms_LSB + ref_termsLSB), [q_i(t), q_j(t)])

In [105]:
for (i, j) in tau:
    display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + color_terms(latex(eta[(i,j)]))))
    display(Math(r'\eta_{' + str(i) + str(j) + r'}^{SB} = ' + color_terms(latex(etaSB[(i,j)]))))
    display(Math(r'\eta_{' + str(i) + str(j) + r'}^{LSB} = ' + color_terms(latex(etaLSB[(i,j)]))))
    print("------------------------------------------------------")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


<IPython.core.display.Math object>

...

------------------------------------------------------


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


## First-generation TDI — X1 combination

We use the standard X1 Michelson combination on the carrier $\eta$ measurements.
The sideband combination $\eta^{SB}$ can be processed analogously.

In [106]:
def P12(expr): return expr - D(expr, tau13 + tau31)
def P21(expr): return D(expr, tau12) - D(expr, tau12 + tau13 + tau31)
def P13(expr): return -(expr - D(expr, tau12 + tau21))
def P31(expr): return -(D(expr, tau13) - D(expr, tau13 + tau12 + tau21))

# X1 using carrier measurements
X1 = collect(expand(
    P13(eta[(1,3)] + D(eta[(3,1)], tau13)) +
    P12(eta[(1,2)] + D(eta[(2,1)], tau12))
), [q1(t), q2(t), q3(t), phi1(t), phi2(t), phi3(t)])

display(Math(r'X_1 = ' + color_terms(latex(X1))))

<IPython.core.display.Math object>

In [107]:
# X1 using sideband measurements
X1_SB = collect(expand(
    P13(etaSB[(1,3)] + D(etaSB[(3,1)], tau13)) +
    P12(etaSB[(1,2)] + D(etaSB[(2,1)], tau12))
), [q1(t), q2(t), q3(t), q_ref(t), phi1(t), phi2(t), phi3(t), phiREF(t)])

display(Math(r'X_1^{SB} = ' + color_terms(latex(X1_SB))))

<IPython.core.display.Math object>

In [110]:
# ── Clock-noise correction variables r_{ij} ────────────────────────────────
# r = -(eta_carrier - eta_sideband) / omega_j^m, keeping only q and phi terms
r = {}
for (i, j) in tau:
    phi_i, q_i, om_i, omm_j_val = sc[i]
    phi_j, q_j, om_j, omm_j = sc[j]
    r[(i, j)] = simplify(-(etaLSB[(i, j)] - etaSB[(i, j)]) / (omm_j+omega_rm)/2)

for (i, j) in tau:
    display(Math(r'r_{' + str(i) + str(j) + r'} / \omega_' + str(j) + r'^m = '
                 + color_terms(latex(r[(i, j)]))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [111]:
a = {
    (1,2): omega1 - omega2,
    (2,1): omega2 - omega1,
    (1,3): omega1 - omega3,
    (3,1): omega3 - omega1,
    (2,3): omega2 - omega3,
    (3,2): omega3 - omega2,
}

R = {}
R[(1,2)] = -(r[(1,3)] + D(r[(3,1)], tau13))
R[(1,3)] =   r[(1,2)] + D(r[(2,1)], tau12)
R[(2,1)] = (r[(1,2)] - r[(1,3)] - D(r[(3,1)], tau13) - D(r[(1,2)], tau13 + tau31))
R[(3,1)] = (-r[(1,3)] + r[(1,2)] + D(r[(2,1)], tau12) + D(r[(1,3)], tau12 + tau21))
R[(2,3)] = 0
R[(3,2)] = 0

triplets = [(1,2,3), (2,3,1), (3,1,2)]
correction = 0
for (i, j, k) in triplets:
    correction -= (- a[(i,j)] * R[(i,j)] - a[(i,k)] * R[(i,k)])

X1c = simplify(X1 - correction)

print("The X1 expression is:")
display(Math(r'X_1 = ' + color_terms(latex(X1))))
print("The corrected X1 expression is:")
display(Math(r'X_1^{corr} = ' + color_terms(latex(X1c))))

The X1 expression is:


<IPython.core.display.Math object>

The corrected X1 expression is:


<IPython.core.display.Math object>